# Notebook 05: Distance Measures & K-Nearest Neighbors
**Lab Assignment:** Assignment 5  
**Project Role:** Vectorized similarity measures, KNN classification, content-based recommendation via TF-IDF cosine distance, and collaborative filtering.


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))
from src.data_loader import load_data
from src.preprocessing import clean_dataframes
from src.distances import pairwise_euclidean, pairwise_manhattan, pairwise_cosine_distance, pairwise_cosine_similarity
from src.knn import KNNClassifierScratch, ContentBasedKNNRecommender
from sklearn.feature_extraction.text import TfidfVectorizer
df_posts, df_interactions = load_data()
df_posts, df_interactions = clean_dataframes(df_posts, df_interactions)
tfidf = TfidfVectorizer(max_features=100, stop_words='english')
post_texts = (df_posts['title'].fillna('') + ' ' + df_posts['abstract'].fillna('')).tolist()
tfidf_mat = tfidf.fit_transform(post_texts).toarray()
recommender = ContentBasedKNNRecommender().fit(df_posts['item_id'].tolist(), tfidf_mat)
sample_history = [df_posts['item_id'].iloc[0], df_posts['item_id'].iloc[1]]
recs = recommender.recommend_for_history(sample_history, k=5)
print(f'User History: {sample_history}')
print('Top 5 Content-Based Recommendations:')
for item, score in recs:
    title = df_posts[df_posts['item_id'] == item]['title'].values[0]
    print(f' - [{item}] (Sim: {score:.3f}): {title}')


In [ ]:
from src.features import SubstackFeaturePipeline
pipeline = SubstackFeaturePipeline().fit(df_posts, df_interactions)
features = pipeline.extract_features(df_posts, df_interactions)
X = features[['word_count', 'item_popularity', 'topic_affinity_score']].values[:500]
y = features['clicked'].values[:500]
knn_scratch = KNNClassifierScratch(n_neighbors=5, metric='cosine', weights='distance').fit(X, y)
knn_sk = KNeighborsClassifier(n_neighbors=5, metric='cosine', weights='distance').fit(X, y)
print('Scratch KNN Accuracy:', np.mean(knn_scratch.predict(X) == y))
print('Sklearn KNN Accuracy:', np.mean(knn_sk.predict(X) == y))


### Interpretation & Findings
1. **Metric Superiority:** For text representation, cosine distance vastly outperforms Euclidean distance because Euclidean distance is distorted by document length.
2. **Cold-Start Content Matching:** Content KNN successfully recommends articles with high topic relevance even when the candidate posts have zero historical clicks.

### Limitations
- Pairwise distance matrix computation scales quadratically $\mathcal{O}(N^2)$ without approximate nearest neighbors (such as HNSW or Annoy).
